# Which Financial Datum Best Predicts a Stock's Annual Return?

**A cross-sectional feature-importance research notebook for QuantConnect.**

This notebook runs entirely in the QuantConnect **Research** environment (`QuantBook`).
It builds a point-in-time panel of fundamental + technical features for a liquid US
equity universe, aligns each observation with the stock's **forward 12-month return**,
and then ranks the features by how well they predict that return.

## The question
Out of dozens of financial data points (valuation, quality, growth, size, leverage,
yield, momentum), **which single one carries the most predictive weight** for the next
year's return?

## Method (three independent lenses, so we don't trust a single number)
1. **Univariate Information Coefficient (IC)** — per-year Spearman rank correlation
   between each feature and forward return. We report the mean IC, its stability
   (Information Ratio = mean/std across years), a t-statistic, and a **Bonferroni-
   corrected** p-value. This is the primary answer to "which single datum matters most."
2. **Standardized linear regression (OLS)** — betas on rank-normalized features,
   showing marginal contribution once features compete with each other.
3. **Random Forest importance** — captures non-linear / interaction effects a linear
   model misses.

A feature that scores high on **all three** is a genuinely robust predictor.

## Bias controls (from the repo's `backtest-expert` methodology)
- **Survivorship bias → avoided.** The universe is rebuilt point-in-time at each annual
  snapshot from QuantConnect's Fundamental universe, which *includes* names later
  delisted/merged/bankrupt. We never seed the study from today's survivors.
- **Look-ahead bias → avoided.** Features are read as of snapshot date `t`; the target
  return is measured strictly *after* `t` (t → t+1y). No future data enters the features.
- **Regime bias → neutralized.** Features and returns are rank-transformed *within each
  year* before pooling, so no single bull/bear year dominates the ranking.
- **Data-mining bias → penalized.** We test many features, so p-values get a Bonferroni
  correction and we demand *sign-consistent* IC across years, not one lucky year.

> Educational research only. Not investment advice. Fundamental predictive power is
> weak and unstable by nature — expect small ICs (|IC| ~ 0.02-0.06 is typical and can
> still be economically meaningful over a large cross-section).

## 1. Setup

In [ ]:
# region imports
from AlgorithmImports import *

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
# endregion

qb = QuantBook()
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('QuantBook ready:', qb.start_date, '->', qb.end_date)

## 2. Configuration

All knobs live here. Kept to round numbers to avoid curve-fitting the *study* itself.

- `SNAPSHOT_YEARS`: annual cross-sections. Each contributes one independent yearly IC.
  8 snapshots gives enough years to compute a stable IC t-stat while covering bull,
  bear (2018, 2022) and recovery regimes.
- `UNIVERSE_SIZE`: top-N by dollar volume among names with valid fundamentals. Focusing
  on liquid names removes micro-cap data-quality noise and keeps the study tradable.
- `HOLD_DAYS`: ~1 trading year for the forward-return target.
- `WINSOR`: clip extreme forward returns (fat tails / bad ticks) at the 1/99 pct.

In [ ]:
SNAPSHOT_MONTH, SNAPSHOT_DAY = 1, 5     # first cross-section date each year
SNAPSHOT_YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
UNIVERSE_SIZE  = 250                    # liquid names per snapshot
HOLD_DAYS      = 365                    # forward-return horizon (~12 months)
MIN_PRICE      = 5.0                    # penny-stock filter (data quality)
WINSOR         = 0.01                   # winsorize forward returns at 1%/99%

snapshot_dates = [datetime(y, SNAPSHOT_MONTH, SNAPSHOT_DAY) for y in SNAPSHOT_YEARS]
print('Snapshots:', [d.strftime('%Y-%m-%d') for d in snapshot_dates])

## 3. The feature set

Each feature is chosen because it has a **documented economic rationale** for predicting
returns (a requirement of the methodology — we don't blindly mine every field).

`candidates` are lowercase substrings used to locate the column in the flattened
fundamental DataFrame, making the code robust to minor QC naming changes. `direction`
is the *hypothesized* sign of the return relationship (for interpretation only; the
measured IC decides the truth).

| Group | Feature | Economic thesis |
|---|---|---|
| Value | earnings_yield, fcf_yield | cheap cash/earnings → higher future return |
| Value | pe_ratio, pb_ratio, ps_ratio | expensive multiples → lower future return |
| Quality | roe, roa, gross_margin, net_margin | profitable firms compound |
| Growth | revenue_growth, eps_growth | growers re-rate |
| Size | log_market_cap | small-cap premium (usually negative sign) |
| Leverage | debt_to_equity | over-levered firms are fragile |
| Yield | dividend_yield | shareholder yield factor |
| Momentum | mom_12_1 | 12-1 month price momentum (technical control) |

In [ ]:
# name -> (list of lowercase substrings to find the flattened column, hypothesized sign)
FUNDAMENTAL_FEATURES = {
    'earnings_yield':  (['valuationratios', 'earningyield'],            +1),
    'fcf_yield':       (['valuationratios', 'fcfyield'],                +1),
    'pe_ratio':        (['valuationratios', 'peratio'],                 -1),
    'pb_ratio':        (['valuationratios', 'pbratio'],                 -1),
    'ps_ratio':        (['valuationratios', 'psratio'],                 -1),
    'dividend_yield':  (['valuationratios', 'trailingdividendyield'],   +1),
    'roe':             (['operationratios', 'roe'],                     +1),
    'roa':             (['operationratios', 'roa'],                     +1),
    'gross_margin':    (['operationratios', 'grossmargin'],             +1),
    'net_margin':      (['operationratios', 'netmargin'],               +1),
    'revenue_growth':  (['operationratios', 'revenuegrowth'],           +1),
    'eps_growth':      (['earningratios',   'dilutedepsgrowth'],        +1),
    'debt_to_equity':  (['operationratios', 'totaldebtequityratio'],    -1),
    'market_cap':      (['marketcap'],                                  -1),  # -> log_market_cap
}
# mom_12_1 is computed from price history, not fundamentals (added later).
HYP_SIGN = {k: v[1] for k, v in FUNDAMENTAL_FEATURES.items()}
HYP_SIGN['mom_12_1'] = +1

## 4. Point-in-time universe

A fundamental universe selection function. In Research, `universe_history` replays this
selection over history using **point-in-time** fundamentals, so each snapshot reflects
the market as it actually was on that date (delisted names included) — this is what
makes the study survivorship-bias free.

In [ ]:
def select_liquid(fundamental):
    filtered = [f for f in fundamental
                if f.has_fundamental_data and f.price > MIN_PRICE and f.market_cap and f.market_cap > 0]
    ranked = sorted(filtered, key=lambda f: f.dollar_volume, reverse=True)
    return [f.symbol for f in ranked[:UNIVERSE_SIZE]]

universe = qb.add_universe(select_liquid)

## 5. Column resolver + inspection

Pull a small flattened fundamental snapshot first, so we can see the exact column names
in *this* QC version and confirm every feature resolves. If a feature prints
`NOT FOUND`, copy the real column name from the printed list into `FUNDAMENTAL_FEATURES`
above and re-run — nothing else needs to change.

In [ ]:
def resolve_column(columns, substrings):
    """First column whose lowercased name contains ALL given substrings."""
    for col in columns:
        low = str(col).lower().replace('_', '')
        if all(s.replace('_', '') in low for s in substrings):
            return col
    return None

probe = qb.universe_history(universe, snapshot_dates[0], snapshot_dates[0] + timedelta(days=6), flatten=True)
print('Flattened snapshot shape:', probe.shape)
print('Sample columns:', list(probe.columns)[:40])
print()
cols = list(probe.columns)
for name, (subs, _sign) in FUNDAMENTAL_FEATURES.items():
    hit = resolve_column(cols, subs)
    print(f'{name:16s} -> {hit if hit else "*** NOT FOUND ***"}')

## 6. Build the panel

For each annual snapshot `t`:
1. Get the point-in-time universe + fundamentals as of `t` (survivorship-safe).
2. Read each fundamental feature via the resolver; `market_cap` → `log_market_cap`.
3. Compute **12-1 momentum**: return from `t-12m` to `t-1m` (price data before `t` only).
4. Compute the **forward 12-month return** (`t` → `t+HOLD_DAYS`) as the prediction target.

Delisted names simply stop having prices — their forward return is measured to the last
available close (capturing the loss), which is the bias-free behavior.

In [ ]:
def price_series(symbols, start, end):
    """Daily close DataFrame indexed by date, columns=symbol, for the given window."""
    if not symbols:
        return pd.DataFrame()
    h = qb.history(symbols, start, end, Resolution.DAILY)
    if h is None or h.empty or 'close' not in h.columns:
        return pd.DataFrame()
    return h['close'].unstack(level=0)  # index=time, columns=symbol


def snapshot_fundamentals(t):
    """Return {symbol: {feature: value}} for the point-in-time universe at date t."""
    fdf = qb.universe_history(universe, t, t + timedelta(days=6), flatten=True)
    if fdf is None or fdf.empty:
        return {}, []
    # take the earliest available trading day in the window as the snapshot
    first_time = fdf.index.get_level_values(-1).min() if fdf.index.nlevels > 1 else fdf.index.min()
    day = fdf.xs(first_time, level=-1) if fdf.index.nlevels > 1 else fdf
    cols = list(day.columns)
    resolved = {name: resolve_column(cols, subs) for name, (subs, _s) in FUNDAMENTAL_FEATURES.items()}

    out = {}
    for sym, row in day.iterrows():
        rec = {}
        for name, col in resolved.items():
            val = row[col] if col is not None and col in row else np.nan
            rec[name] = float(val) if pd.notna(val) else np.nan
        # transform market_cap -> log_market_cap
        if 'market_cap' in rec:
            mc = rec.pop('market_cap')
            rec['log_market_cap'] = np.log(mc) if (mc and mc > 0) else np.nan
        out[sym] = rec
    return out, list(out.keys())

In [ ]:
rows = []
for t in snapshot_dates:
    funda, symbols = snapshot_fundamentals(t)
    if not symbols:
        print(f'{t:%Y-%m-%d}: no universe data, skipping')
        continue

    # forward 12m return target (t -> t+HOLD_DAYS)
    fwd_px = price_series(symbols, t, t + timedelta(days=HOLD_DAYS + 7))
    # 12-1 momentum feature (t-365 -> t-30), strictly before t
    mom_px = price_series(symbols, t - timedelta(days=365), t)

    for sym in symbols:
        feats = dict(funda[sym])

        # forward return
        fwd_ret = np.nan
        if sym in fwd_px.columns:
            s = fwd_px[sym].dropna()
            if len(s) >= 2:
                fwd_ret = s.iloc[-1] / s.iloc[0] - 1.0

        # 12-1 momentum
        mom = np.nan
        if sym in mom_px.columns:
            s = mom_px[sym].dropna()
            if len(s) >= 20:
                mom = s.iloc[-21] / s.iloc[0] - 1.0  # skip most recent ~1 month
        feats['mom_12_1'] = mom

        if pd.notna(fwd_ret):
            feats['year'] = t.year
            feats['symbol'] = str(sym)
            feats['fwd_return'] = fwd_ret
            rows.append(feats)
    print(f'{t:%Y-%m-%d}: {sum(1 for r in rows if r["year"] == t.year)} usable observations')

panel = pd.DataFrame(rows)
print('\nPanel shape:', panel.shape)
panel.head()

## 7. Clean & normalize (per-year, cross-sectional)

To neutralize regime effects we rank-transform **within each year**: every feature is
converted to a within-year percentile (0-1), and the forward return is winsorized then
also handled per year in the IC step. Rank-based features make the study robust to
outliers and non-linear scaling — exactly what Spearman IC expects.

In [ ]:
FEATURES = [c for c in panel.columns if c not in ('year', 'symbol', 'fwd_return')]
print('Features under test:', FEATURES)

# winsorize forward return within each year
def winsorize(s, p):
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

panel['fwd_return_w'] = panel.groupby('year')['fwd_return'].transform(lambda s: winsorize(s, WINSOR))

# per-year percentile rank of each feature (0..1); NaNs stay NaN
ranked = panel.copy()
for f in FEATURES:
    ranked[f] = panel.groupby('year')[f].transform(lambda s: s.rank(pct=True))

coverage = panel[FEATURES].notna().mean().sort_values()
print('\nFeature coverage (non-NaN fraction):')
print(coverage.to_string())

## 8. Lens 1 — Univariate Information Coefficient (primary answer)

For each feature and each year, the Spearman rank correlation with the forward return.
Aggregated across years we report:
- **mean_IC** — average predictive correlation (the headline number).
- **IC_IR** — mean/std across years (Information Ratio): *stability*, not just strength.
- **t_stat / p_value** — is mean IC different from zero?
- **p_bonferroni** — p-value × number of features (guards against data mining).
- **sign_consistency** — fraction of years with the dominant sign (robustness check).

The feature with the largest `|mean_IC|` that is **Bonferroni-significant** and
**sign-consistent** is the datum that most reliably predicts annual return.

In [ ]:
n_features = len(FEATURES)
records = []
per_year_ic = {}

for f in FEATURES:
    ics = []
    for yr, g in panel.groupby('year'):
        sub = g[[f, 'fwd_return_w']].dropna()
        if len(sub) >= 20:
            ic, _ = stats.spearmanr(sub[f], sub['fwd_return_w'])
            if pd.notna(ic):
                ics.append(ic)
    per_year_ic[f] = ics
    if len(ics) >= 3:
        ics_arr = np.array(ics)
        mean_ic = ics_arr.mean()
        std_ic = ics_arr.std(ddof=1)
        ic_ir = mean_ic / std_ic if std_ic > 0 else np.nan
        t_stat, p_val = stats.ttest_1samp(ics_arr, 0.0)
        dom_sign = np.sign(mean_ic)
        sign_consistency = np.mean(np.sign(ics_arr) == dom_sign)
        records.append({
            'feature': f,
            'mean_IC': mean_ic,
            'abs_IC': abs(mean_ic),
            'IC_IR': ic_ir,
            't_stat': t_stat,
            'p_value': p_val,
            'p_bonferroni': min(1.0, p_val * n_features),
            'sign_consistency': sign_consistency,
            'n_years': len(ics),
        })

ic_table = pd.DataFrame(records).sort_values('abs_IC', ascending=False).reset_index(drop=True)
ic_table

In [ ]:
# Visualize mean |IC| with Bonferroni significance highlighted
fig, ax = plt.subplots(figsize=(10, 6))
tbl = ic_table.sort_values('abs_IC')
colors = ['#2ca02c' if p < 0.05 else '#c7c7c7' for p in tbl['p_bonferroni']]
ax.barh(tbl['feature'], tbl['abs_IC'], color=colors)
ax.set_xlabel('|mean Information Coefficient|  (Spearman, per-year averaged)')
ax.set_title('Univariate predictive power of each financial datum\n(green = Bonferroni-significant p<0.05)')
for i, (_, r) in enumerate(tbl.iterrows()):
    ax.text(r['abs_IC'], i, f"  IR={r['IC_IR']:.2f}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Per-year IC heat-style table for the top features (stability check)
top_feats = ic_table.head(8)['feature'].tolist()
stab = pd.DataFrame({f: pd.Series(per_year_ic[f]) for f in top_feats})
stab.index = [f'IC_{i+1}' for i in range(len(stab))]
print('Per-year IC for the top-8 features (one column per feature):')
stab.T

## 9. Lens 2 — Standardized linear regression

Pool all years using the per-year **ranked** features (already 0-1, comparable scale) and
regress the ranked forward return on them jointly. The coefficient magnitude is the
feature's marginal contribution when competing against the others (controls for the fact
that, e.g., value and size are correlated).

In [ ]:
# ranked forward return target (per-year percentile), and ranked features
ranked['fwd_rank'] = panel.groupby('year')['fwd_return_w'].transform(lambda s: s.rank(pct=True))
reg = ranked[FEATURES + ['fwd_rank']].dropna()
print('Regression sample:', reg.shape)

X = reg[FEATURES].values
y = reg['fwd_rank'].values
lin = LinearRegression().fit(X, y)
ols_tbl = pd.DataFrame({'feature': FEATURES, 'ols_coef': lin.coef_})
ols_tbl['abs_coef'] = ols_tbl['ols_coef'].abs()
ols_tbl = ols_tbl.sort_values('abs_coef', ascending=False).reset_index(drop=True)
print(f'R^2 = {lin.score(X, y):.4f}')
ols_tbl

## 10. Lens 3 — Random Forest importance

A non-linear model. Feature importances capture predictive value from interactions and
non-monotonic relationships that IC and OLS would miss. Fit on the same pooled ranked
panel.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=50,
    random_state=42, n_jobs=-1,
)
rf.fit(X, y)
rf_tbl = pd.DataFrame({'feature': FEATURES, 'rf_importance': rf.feature_importances_})
rf_tbl = rf_tbl.sort_values('rf_importance', ascending=False).reset_index(drop=True)
rf_tbl

## 11. Combined ranking — the verdict

Rank each feature 1..N under all three lenses (|IC|, |OLS coef|, RF importance) and
average the ranks. A low average rank = a datum that predicts forward return robustly
across *linear, non-linear, and univariate* views. The IC table above is still the
headline (it's the most interpretable and regime-controlled); this consensus view is the
tie-breaker and robustness confirmation.

In [ ]:
merged = (ic_table[['feature', 'mean_IC', 'abs_IC', 'IC_IR', 'p_bonferroni', 'sign_consistency']]
          .merge(ols_tbl[['feature', 'abs_coef']], on='feature', how='outer')
          .merge(rf_tbl, on='feature', how='outer'))

merged['rank_IC'] = merged['abs_IC'].rank(ascending=False)
merged['rank_OLS'] = merged['abs_coef'].rank(ascending=False)
merged['rank_RF'] = merged['rf_importance'].rank(ascending=False)
merged['consensus_rank'] = merged[['rank_IC', 'rank_OLS', 'rank_RF']].mean(axis=1)
verdict = merged.sort_values('consensus_rank').reset_index(drop=True)
verdict

In [ ]:
winner = verdict.iloc[0]
direction = 'higher' if HYP_SIGN.get(winner['feature'], np.sign(winner['mean_IC'])) * winner['mean_IC'] >= 0 else 'lower'
sign_word = 'positively' if winner['mean_IC'] >= 0 else 'negatively'
print('=' * 64)
print('MOST PREDICTIVE FINANCIAL DATUM (consensus across 3 lenses):')
print(f"  -> {winner['feature'].upper()}")
print('=' * 64)
print(f"  mean IC          : {winner['mean_IC']:+.4f}  ({sign_word} related to next-year return)")
print(f"  IC Info Ratio    : {winner['IC_IR']:+.2f}   (stability across years)")
print(f"  Bonferroni p     : {winner['p_bonferroni']:.4f}")
print(f"  sign consistency : {winner['sign_consistency']:.0%} of years")
print(f"  RF importance    : {winner['rf_importance']:.4f}")
print()
print('Interpretation: stocks ranking high on this metric tended to deliver',
      'higher' if winner['mean_IC'] >= 0 else 'lower', 'forward 12-month returns.')
print('Caveat: fundamental ICs are small and unstable; confirm with a full backtest',
      '(see repo backtest-expert skill) before trading on it.')

## 12. Persist results

Save the tables so they can be diffed / tracked over time (matches the repo's
`reports/` convention).

In [ ]:
import os
os.makedirs('reports', exist_ok=True)
stamp = f"{SNAPSHOT_YEARS[0]}_{SNAPSHOT_YEARS[-1]}"
ic_table.to_csv(f'reports/feature_ic_{stamp}.csv', index=False)
verdict.to_csv(f'reports/feature_consensus_{stamp}.csv', index=False)
print('Saved reports/feature_ic_*.csv and reports/feature_consensus_*.csv')
print('\nTop 5 predictors:')
print(verdict.head(5)[['feature', 'mean_IC', 'IC_IR', 'p_bonferroni', 'consensus_rank']].to_string(index=False))

## 13. How to extend

- **More features:** add rows to `FUNDAMENTAL_FEATURES` (run cell 5 to confirm the
  column resolves). Any Morningstar field works.
- **Different horizon:** change `HOLD_DAYS` (e.g. 63 for quarterly, 21 for monthly) and
  add more `SNAPSHOT_YEARS`/intra-year snapshots for a larger sample.
- **Sector-neutral:** rank features within (year, sector) instead of just year to strip
  out sector bets (add `morningstar_sector_code` as a feature and group by it).
- **Confirm economically:** promote the winning datum to a long/short decile backtest as
  a `QCAlgorithm` (see `lean-backtests/` for the template) before trusting it. IC is
  necessary but not sufficient — friction and turnover can erase a real signal.